In [1]:
import pandas as pd
import numpy as np
from jupyter_ruler import RuleStrategyApp

In [2]:
# 生成示例数据
np.random.seed(0)
n = 5000
dates = pd.date_range('2023-01-01', '2024-12-31', periods=n)
df = pd.DataFrame({
    # 用户行为特征
    'query_count': np.random.randint(1, 50, n),  # 查询次数
    'overdue_count': np.random.randint(0, 10, n),  # 逾期次数
    'account_count': np.random.randint(1, 20, n),  # 账户数量
    'page_views': np.random.randint(5, 200, n),  # 界面浏览次数
    'click_count': np.random.randint(10, 500, n),  # 界面按键点击次数
    'product_count': np.random.randint(1, 15, n),  # 产品数量
    'credit_history': np.random.randint(0, 30, n),  # 历史授信次数
    
    # 财务特征
    'income': np.random.normal(8000, 3000, n).clip(2000, 20000),  # 收入
    'debt_ratio': np.random.uniform(0.1, 0.8, n),  # 负债率
    'credit_score': np.random.randint(300, 850, n),  # 信用评分
    
    # 申请时间
    'apply_time': dates,
})

In [3]:
# 衍生标签
# 客群评级 A > B > C > D
df['customer_rating'] = np.random.choice(['A', 'B', 'C', 'D'], n, p=[0.25, 0.35, 0.28, 0.12])

# 通过标签: 1=通过, 2=拒绝 (目标通过率70%)
df['pass_label'] = np.random.choice([1, 2], n, p=[0.70, 0.30])

# 转人工标签: 1=通过, 2=拒绝, 空=未转人工
# 约5%的申请会转人工审核，转人工通过率60%
manual_prob = 0.05
manual_pass_prob = 0.6
manual_status = []
for _ in range(n):
    if np.random.random() < manual_prob:
        manual_status.append(1 if np.random.random() < manual_pass_prob else 2)
    else:
        manual_status.append(np.nan)
df['manual_review'] = manual_status

# 逾期标签1: MOB3 30+ (逾期30天以上)
# 逾期标签2: MOB6 60+ (逾期60天以上)
# 目标坏样本率约5%
df['overdue_mob3'] = np.random.choice([0, 1], n, p=[0.96, 0.04])
df['overdue_mob6'] = np.random.choice([0, 1], n, p=[0.98, 0.02])

# 产品信息
products = ['信用贷', '消费贷', '经营贷', '车贷', '房贷']
product_weights = [0.35, 0.25, 0.15, 0.15, 0.10]
df['product_name'] = np.random.choice(products, n, p=product_weights)
df['term'] = np.random.choice([6, 12, 24, 36, 48, 60], n, p=[0.1, 0.25, 0.3, 0.2, 0.1, 0.05])
df['interest_rate'] = np.round(np.random.uniform(0.03, 0.15, n), 3)
df['loan_amount'] = np.random.randint(5000, 500000, n)

# 目标变量 (1=坏样本, 0=好样本) - 基于逾期标签
df['target'] = (df['overdue_mob3'] | df['overdue_mob6']).astype(int)

# 申请通过时间和放款时间 (基于申请时间)
df['pass_time'] = df['apply_time'] + pd.to_timedelta(np.random.randint(0, 72, n), unit='h')
df['disburse_time'] = df['pass_time'] + pd.to_timedelta(np.random.randint(0, 48, n), unit='h')

In [4]:
# 筛选已通过的样本用于分析 (只有通过的样本才需要策略监控)
df_approved = df[df['pass_label'] == 1].reset_index(drop=True)

print(f"总申请样本: {len(df)}")
print(f"通过样本: {len(df_approved)}")
print(f"通过率: {len(df_approved)/len(df):.1%}")
print(f"坏样本率: {df_approved['target'].mean():.2%}")
print(f"\n转人工审核数: {df['manual_review'].notna().sum()}")
print(f"转人工率: {df['manual_review'].notna().mean():.1%}")
print(f"\n数据时间范围: {df['apply_time'].min()} 至 {df['apply_time'].max()}")
df_approved.head()

总申请样本: 5000
通过样本: 3520
通过率: 70.4%
坏样本率: 5.65%

转人工审核数: 223
转人工率: 4.5%

数据时间范围: 2023-01-01 00:00:00 至 2024-12-31 00:00:00


,query_count,overdue_count,account_count,page_views,click_count,product_count,credit_history,income,debt_ratio,credit_score,...,manual_review,overdue_mob3,overdue_mob6,product_name,term,interest_rate,loan_amount,target,pass_time,disburse_time
0,48,7,15,164,24,2,8,7325.836230,0.703490,396,...,NaN,0,0,车贷,24,0.035,161821,0,2023-01-02 10:30:16.923384676,2023-01-02 14:30:16.923384676
1,4,7,10,67,124,2,7,2000.000000,0.658113,646,...,NaN,0,0,车贷,12,0.115,214044,0,2023-01-03 14:30:50.770154030,2023-01-04 00:30:50.770154030
2,4,8,5,111,489,8,6,8456.050364,0.511027,324,...,NaN,0,0,消费贷,24,0.134,460993,0,2023-01-01 19:01:07.693538707,2023-01-02 22:01:07.693538707
3,40,9,4,109,203,5,25,7809.210450,0.370838,363,...,NaN,0,0,信用贷,6,0.034,40425,0,2023-01-04 16:31:24.616923384,2023-01-05 13:31:24.616923384
4,10,4,6,48,488,1,6,6155.443920,0.439407,319,...,NaN,0,0,信用贷,6,0.104,129063,0,2023-01-04 01:01:41.540308061,2023-01-04 11:01:41.540308061


In [6]:
# 创建策略分析应用
# 使用行为特征和财务特征作为规则挖掘的输入
feature_cols = [
    'query_count', 'overdue_count', 'account_count',
    'page_views', 'click_count', 'product_count', 'credit_history',
    'income', 'debt_ratio', 'credit_score'
]

app = RuleStrategyApp(
    df=df,
    feature_cols=feature_cols,
    target  ='target',
    date_col='apply_time'
)
app.display()